# Crop Yield Analysis Dashboard

This notebook provides comprehensive analysis of crop yield data with interactive visualizations and statistical insights.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for matplotlib
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Load and Explore Crop Yield Dataset

In [ ]:
# Load crop yield dataset
crop_data = pd.read_csv('crop_yield_dataset.csv')

print("Dataset Shape:", crop_data.shape)
print("\nColumn Names:", crop_data.columns.tolist())
print("\nFirst 5 rows:")
print(crop_data.head())

print("\nDataset Info:")
print(crop_data.info())

print("\nBasic Statistics:")
print(crop_data.describe())

## Data Preprocessing and Cleaning

In [ ]:
# Check for missing values
print("Missing values:")
print(crop_data.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {crop_data.duplicated().sum()}")

# Check data types
print("\nData types:")
print(crop_data.dtypes)

# Check unique values for categorical columns
categorical_cols = ['soil_type', 'weather_condition', 'crop_type']
for col in categorical_cols:
    print(f"\nUnique values in {col}: {crop_data[col].unique()}")
    
# Handle any outliers (using IQR method for crop_yield)
Q1 = crop_data['crop_yield'].quantile(0.25)
Q3 = crop_data['crop_yield'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nCrop yield outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
outliers = crop_data[(crop_data['crop_yield'] < lower_bound) | (crop_data['crop_yield'] > upper_bound)]
print(f"Number of outliers: {len(outliers)}")

# Clean data (remove outliers for better analysis)
crop_data_clean = crop_data[(crop_data['crop_yield'] >= lower_bound) & (crop_data['crop_yield'] <= upper_bound)].copy()
print(f"Dataset shape after cleaning: {crop_data_clean.shape}")

## Crop Yield Statistical Analysis

In [ ]:
# Correlation analysis
numerical_cols = ['temperature', 'rainfall', 'humidity', 'crop_yield']
correlation_matrix = crop_data_clean[numerical_cols].corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Yield analysis by categorical variables
print("\n=== YIELD ANALYSIS BY CROP TYPE ===")
yield_by_crop = crop_data_clean.groupby('crop_type')['crop_yield'].agg(['mean', 'median', 'std', 'count']).round(2)
print(yield_by_crop)

print("\n=== YIELD ANALYSIS BY SOIL TYPE ===")
yield_by_soil = crop_data_clean.groupby('soil_type')['crop_yield'].agg(['mean', 'median', 'std', 'count']).round(2)
print(yield_by_soil)

print("\n=== YIELD ANALYSIS BY WEATHER CONDITION ===")
yield_by_weather = crop_data_clean.groupby('weather_condition')['crop_yield'].agg(['mean', 'median', 'std', 'count']).round(2)
print(yield_by_weather)

# Environmental factor impact
print("\n=== ENVIRONMENTAL FACTOR RANGES ===")
for col in ['temperature', 'rainfall', 'humidity']:
    print(f"{col.capitalize()}:")
    print(f"  Range: {crop_data_clean[col].min():.2f} - {crop_data_clean[col].max():.2f}")
    print(f"  Mean: {crop_data_clean[col].mean():.2f}")
    print(f"  Std: {crop_data_clean[col].std():.2f}")
    print()

## Crop Yield Visualization

In [ ]:
# Create comprehensive visualization dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Crop Yield Distribution', 'Correlation Heatmap',
        'Yield by Crop Type', 'Environmental Factors vs Yield',
        'Yield by Soil Type', 'Yield by Weather Condition'
    ],
    specs=[[{"type": "histogram"}, {"type": "heatmap"}],
           [{"type": "bar"}, {"type": "scatter"}],
           [{"type": "box"}, {"type": "violin"}]]
)

# 1. Crop yield distribution
fig.add_trace(
    go.Histogram(x=crop_data_clean['crop_yield'], name='Yield Distribution', nbinsx=30),
    row=1, col=1
)

# 2. Correlation heatmap
fig.add_trace(
    go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=correlation_matrix.values.round(3),
        texttemplate="%{text}",
        showscale=False
    ),
    row=1, col=2
)

# 3. Average yield by crop type
avg_yield_crop = crop_data_clean.groupby('crop_type')['crop_yield'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=avg_yield_crop.index, y=avg_yield_crop.values, name='Avg Yield by Crop'),
    row=2, col=1
)

# 4. Temperature vs Yield scatter
fig.add_trace(
    go.Scatter(
        x=crop_data_clean['temperature'], 
        y=crop_data_clean['crop_yield'],
        mode='markers',
        name='Temperature vs Yield',
        opacity=0.6
    ),
    row=2, col=2
)

# 5. Yield by soil type (box plot)
for soil in crop_data_clean['soil_type'].unique():
    soil_data = crop_data_clean[crop_data_clean['soil_type'] == soil]['crop_yield']
    fig.add_trace(
        go.Box(y=soil_data, name=soil, showlegend=False),
        row=3, col=1
    )

# 6. Yield by weather condition (violin plot)
for weather in crop_data_clean['weather_condition'].unique():
    weather_data = crop_data_clean[crop_data_clean['weather_condition'] == weather]['crop_yield']
    fig.add_trace(
        go.Violin(y=weather_data, name=weather, showlegend=False),
        row=3, col=2
    )

# Update layout
fig.update_layout(
    height=1200,
    title_text="Comprehensive Crop Yield Analysis Dashboard",
    title_x=0.5,
    showlegend=True
)

fig.show()